# How many simulations are enough? A short walkthrough

This notebook reproduces the core idea of the article in a few cells. The full experiments are in `scripts/run_experiments.py`.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
from mcsize import stats, withdrawal as wd

## 1. The rule of thumb

For an error below $1/x$, run at least $x^2$ simulations. For a probability, the 95% half-width is $1.96\sqrt{p(1-p)/N}$, and $p(1-p)\le 1/4$, so with $N=x^2$ the half-width is at most $0.98/x$.

In [ ]:
for x in (10, 100, 1000):
    n = stats.n_rule_of_thumb(1 / x)
    print(x, n, round(stats.zval() * np.sqrt(0.25 / n), 6))

## 2. One simulation of the 4% withdrawal plan

In [ ]:
rng = np.random.default_rng(1)
scen = wd.draw_scenarios(1000, rng, 'iid')
balance, failed = wd.evaluate(scen, 0.04)
k = int(failed.sum())
print(k, 'failures out of 1,000:', stats.wald(k, 1000))

## 3. How many runs for a target precision?

In [ ]:
p = 0.0633
print('±0.5 point :', stats.n_for_prob_abs(p, 0.005))
print('±10% of p  :', stats.n_for_prob_rel(p, 0.10))
print('p = 0.001, ±10% of p:', stats.n_for_prob_rel(0.001, 0.10))

## 4. Two plans on the same scenarios

In [ ]:
scen = wd.draw_scenarios(100_000, rng, 'iid')
_, fa = wd.evaluate(scen, 0.04)
_, fb = wd.evaluate(scen, 0.0425)
d = fb.astype(float) - fa
print('difference', d.mean(), '±', stats.zval() * d.std(ddof=1) / np.sqrt(len(d)))